# Goal

Берём лучшего агента `:160` с `42.4`, крутим PPO крутилки, смотрим, что получается. Надежда, что можно получить супер-чемпиона.

# TARGET_NOTEBOOK_FNAME


In [1]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_07_old.ipynb'

# OPTUNA_DECISIVE_METRIC


In [2]:
OPTUNA_DECISIVE_METRIC = 'test/levels_passed_mean'

# GRID_SEARCH_SPACE


In [3]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters


In [4]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = random.randint(1, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True

    HP.env.count = 32
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    wm_parent = '18d_world_model_09:183'
    HP.vision_head.parent = dict(model=wm_parent, weights=wm_parent)
    HP.vision_head.is_trainable = False

    HP.encoder.parent = dict(model=wm_parent, weights=wm_parent)
    HP.encoder.is_trainable = True

    HP.agent.parent = None
    HP.agent.sequence_length = 4
    HP.agent.action_plan_length = 10
    HP.agent.d_model = 256
    HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.is_trainable = True

    parent = '18n_ppo_tr_frostbite_07_old:160'
    HP.encoder.parent['weights'] = parent
    HP.agent.parent = parent

    HP.test.env_rams = None
    HP.test.env_ram_patches = None
    HP.test.break_on_level_passed = False

    # Rollout: keep nine_lives (deep chains) + natural play.
    HP.ppo.rollout_env_rams = None
    HP.ppo.rollout_env_ram_patches = [
        ['nine_lives'],
    ]

    # Training procedure params (PPO related)
    HP.ppo.global_steps_count = 3_000_000
    HP.ppo.rollout_steps_count = 512
    HP.ppo.epochs_count = 2
    HP.ppo.batch_size = 512
    HP.ppo.optimizer = 'AdamW'

    HP.ppo.vf_coef = 0.2
    HP.ppo.consistency_coef = 0.1
    HP.ppo.prediction_coef = 0.1

    learn_rate = optuna_trial.suggest_float('learn_rate', 0.000025, 0.00025)
    HP.ppo.learn_rate = f'const({learn_rate})'
    ent_coef = optuna_trial.suggest_float('ent_coef', 0.005, 0.05)
    HP.ppo.ent_coef = f'const({ent_coef})'

    HP.ppo.gamma = 0.997
    HP.ppo.gae_lambda = 0.95
    clip_coef = optuna_trial.suggest_float('clip_coef', 0.05, 0.2)
    HP.ppo.clip_coef = clip_coef
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5
    HP.ppo.target_kl = None
    HP.ppo.norm_adv = True

    return HP

# Results

Видно, что всё равно слабее чем `25.5`, но уже не сильно. Т.е. если сфокусироваться на чемпионе и не распылаться (как это было в `42.5b`), то можно выдавить больше результата. Но правда для этого чемпион должен быть явным. `07_old:160` явно выделяется от своих собратьев, поэтому и его потомки сильнее. 

Одна из мыслей: возможно не хватает плавного аннилинга. Я исходил из того, что аннилинг проще делать ступенчато: на первом шаге такие-то параметры, на втором - поменьше и т.д. Но видимо 3млн шагов - это слишком много долго. Ведь как можно думать об аннилинге. Типа ты выпустил популяцию на многомерную сковородку и подогреваешь её. На этой сковородке есть много развилок, куда можно двигаться: кто-то выбрал одно направление развития, кто-то другое. При этом ты держишь температуру высокой - агенты могут менять своё мнение (сейчас хочу идту в одну стороу, через некоторое время в другую). Но если не понижать температуру, то агенты просто будут вот так скакать между этими выборами и НЕ коммититься (как буриданов осёл). А если температуру начать отпускать, то агенты должны будут специализироваться по выбранному направлению и их будет дальше и дальше друг от друга разносить. Короче, кажется, надо сковородку плавно остужать.

|что|`18n_study_42.5c`|`18n_study_25.5`|
|---|---|---|
|levels_passed_mean|<img src="./img/levels_passed_mean_42.5c.png">|<img src="./img/levels_passed_mean_25.5.png">|
|levels_passed|<img src="./img/levels_passed_42.5c.png">|<img src="./img/levels_passed_25.5.png">|
|episode_lp|<img src="./img/episode_lp_42.5c.png">|n/a|

**Выводы**
1) попробовать плавный аннилинг, но чего?

# System

In [5]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [6]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07_old.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_07_old',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_42.5c/18n_study_42.5c.ipynb',
 'optuna_study_name': '18n_study_42.5c',
 'optuna_study_serial': '42.5c',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_42.5c/18n_study_42.5c.optuna'}

In [7]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [8]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [9]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC=OPTUNA_DECISIVE_METRIC,
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [10]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [11]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 50
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-10-14 23:44:42,792] A new study created in Journal with name: 18n_study_42.5c


2026.10.14-23:44:43.148152     0.330 >> Model instance registered, version=178
2026.10.14-23:44:43.158861     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07_old-launch178.ipynb"
2026.10.14-23:44:43.159229     0.004 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07_old:178"; running launches=1
2026.10.14-23:45:14.945591     0.535 >> Model instance registered, version=179
2026.10.14-23:45:14.954295     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07_old-launch179.ipynb"
2026.10.14-23:45:14.954533     0.004 >> 5.3 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07_old:179"; running launches=2
2026.10.14-23:45:46.699707     0.391 >> Model instance registered, version=180
2026.10.14-23:45:46.714835     0.006 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07_old-launch180.ipynb"
2026.10.14-23:45:46.715549     0.007 >> 4.5 idle runners exist, submitted launch "18n

In [12]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-10-15 06:54:47,519] Using an existing study with name '18n_study_42.5c' instead of creating a new one.


2026.10.15-06:54:47.520488     5.033 >> Study statistics: 
2026.10.15-06:54:47.523789     0.003 >> 	Number of finished trials: 50
2026.10.15-06:54:47.524772     0.001 >> 	Number of pruned trials: 0
2026.10.15-06:54:47.525610     0.001 >> 	Number of complete trials: 50
2026.10.15-06:54:47.526606     0.001 >> Best trial:
2026.10.15-06:54:47.527185     0.001 >> 	Value: 4.4375
2026.10.15-06:54:47.527564     0.000 >> 	Model version: 200
2026.10.15-06:54:47.527925     0.000 >> 	Params: 
2026.10.15-06:54:47.528236     0.000 >> 		learn_rate: 3.207657610218081e-05
2026.10.15-06:54:47.528599     0.000 >> 		ent_coef: 0.0053515120879783105
2026.10.15-06:54:47.528934     0.000 >> 		clip_coef: 0.12168905535906355


In [13]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.10.15-06:54:47.536148     0.007 >> Top trials:
2026.10.15-06:54:47.536776     0.001 >> 1) 18n_ppo_tr_frostbite_07_old:200, value=4.4375
2026.10.15-06:54:47.537124     0.000 >> 2) 18n_ppo_tr_frostbite_07_old:197, value=4.375
2026.10.15-06:54:47.537446     0.000 >> 3) 18n_ppo_tr_frostbite_07_old:224, value=4.375


In [14]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()

(np.float64(3.920625), np.float64(0.4350022449654714))

In [22]:
f'{3.207657610218081e-05:.6f}'

'0.000032'